# Proyecto final: análisis de reseñas de videojuegos con LLMs
Pipeline reproducible en dos etapas: filtrado semántico y extracción estructurada con DeepSeek.

## 1. Objetivo y arquitectura
El notebook presenta el proceso; la lógica reutilizable y validada reside en `src/videogame_reviews`. Cada reseña conserva un identificador estable y un hash de contenido. Los duplicados permanecen en la muestra, pero solo generan una llamada.

## 2. Configuración
`RUN_API=False` permite ejecutar el notebook sin consumir la API. Para una ejecución real, configure `DEEPSEEK_API_KEY` en el entorno y cambie el indicador a `True`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path('src').resolve()))
from videogame_reviews import PipelineConfig, run_pipeline
from videogame_reviews.data import explore_reviews, load_reviews, prepare_sample
from videogame_reviews.evaluation import (
    build_audit_sample, estimate_execution_plan, quality_metrics,
)
from videogame_reviews.prompts import (
    EXTRACTION_PROMPT_VERSION, FILTER_PROMPT_VERSION,
    EXTRACTION_SYSTEM_PROMPT, FILTER_SYSTEM_PROMPT,
)

RUN_API = False
config = PipelineConfig()
print('Modo:', 'API real' if RUN_API else 'sin llamadas API')
print('Credencial configurada:', bool(os.environ.get(config.api_key_env)))

## 3. Carga del dataset

In [ ]:
data = load_reviews(config.input_path)
print(f'Forma: {data.shape}')
display(data.head())

## 4. Exploración
Se revisan dimensiones, tipos, valores nulos y distribución de la valoración.

In [ ]:
eda = explore_reviews(data)
display(pd.DataFrame({'nulos': eda['nulls']}))
display(pd.Series(eda['ratings'], name='cantidad').to_frame())
print(json.dumps({k: eda[k] for k in ('rows', 'null_content', 'dtypes')}, ensure_ascii=False, indent=2))

## 5. Preprocesamiento y top 100
Se eliminan contenidos nulos, se calcula la longitud, se seleccionan las 100 reseñas más largas con desempate determinista y se añade SHA-256.

In [ ]:
preprocessed = run_pipeline(config, stage='preprocess')
sample = preprocessed['sample']
print({
    'filas_muestra': len(sample),
    'contenidos_unicos': sample['content_hash'].nunique(),
    'duplicados_cacheables': sample.duplicated('content_hash').sum(),
    'longitud_minima': sample['longitud_caracteres'].min(),
    'longitud_maxima': sample['longitud_caracteres'].max(),
})
display(sample.head())

## 6. Proveedor, modelos y limitaciones
Se usa la API oficial de DeepSeek mediante el SDK compatible con OpenAI. `deepseek-v4-flash` filtra en lotes de 5 por su menor coste y latencia; `deepseek-v4-pro` extrae ocho atributos en lotes de 3 para priorizar consistencia semántica. El modo de razonamiento se desactiva porque ambas tareas exigen JSON controlado, no razonamiento abierto. La entrada se estima con `ceil(caracteres_del_prompt/4)` y la salida se limita a 2.000/4.000 tokens. Las llamadas son secuenciales y no requieren pausa fija tras un éxito. Los errores 429/500/503, timeouts, JSON inválido y contenido vacío se reintentan hasta 5 veces con backoff de 2, 4, 8 y 16 segundos, limitado a 30. Si se agota un límite o saldo, los checkpoints conservan cada lote validado y la siguiente ejecución continúa sin repetirlo.

In [ ]:
display(pd.DataFrame([
    {'etapa': 'filtrado', 'modelo': config.filter_model, 'batch': config.filter_batch_size, 'max_tokens_salida': config.filter_max_tokens},
    {'etapa': 'extracción', 'modelo': config.extraction_model, 'batch': config.extraction_batch_size, 'max_tokens_salida': config.extraction_max_tokens},
]))
print('Intentos máximos:', config.max_attempts, '| timeout:', config.request_timeout_seconds, 's')

## 7. Prompt de filtrado

In [ ]:
print('Versión:', FILTER_PROMPT_VERSION)
print(FILTER_SYSTEM_PROMPT)

## 8. Ejecución y resultado del filtrado

In [ ]:
if RUN_API:
    pipeline_result = run_pipeline(config, stage='all')
    filtered = pipeline_result['filtered']
    relevant = pipeline_result['relevant']
elif Path('outputs/relevance_results.csv').exists():
    filtered = pd.read_csv('outputs/relevance_results.csv')
    relevant = pd.read_csv('outputs/relevant_reviews.csv')
else:
    filtered = relevant = pd.DataFrame()
    print('Sin resultados de filtrado: ejecute una vez con RUN_API=True.')

if not filtered.empty:
    display(filtered['motivo'].value_counts(dropna=False).to_frame('cantidad'))
    print('Reseñas relevantes:', len(relevant))
    display(filtered[['sample_rank', 'Valoración', 'relevante', 'motivo']].head(10))

## 9. Prompt de extracción

In [ ]:
print('Versión:', EXTRACTION_PROMPT_VERSION)
print(EXTRACTION_SYSTEM_PROMPT)

## 10. Resultado estructurado
La extracción se aplica exclusivamente al DataFrame filtrado. Las listas se exportan como JSON válido dentro del CSV.

In [ ]:
structured_path = Path('outputs/structured_reviews.csv')
if structured_path.exists():
    structured = pd.read_csv(structured_path)
    display(structured.head())
else:
    structured = pd.DataFrame()
    print('Sin extracción real todavía. El CSV proporcionado se muestra solo como referencia de formato:')
    reference = pd.read_csv('analisis_videojuegos_resultados.csv')
    display(reference.head(3))

## 11. Métricas, llamadas y recuperación

In [ ]:
metrics_path = Path('outputs/run_metrics.json')
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    display(pd.json_normalize(metrics).T.rename(columns={0: 'valor'}))
else:
    print('Métricas disponibles después de ejecutar el pipeline.')

if not relevant.empty:
    execution_plan = estimate_execution_plan(
        sample, relevant, config.filter_batch_size, config.extraction_batch_size
    )
    display(pd.Series(execution_plan, name='valor').to_frame())
    print('Llamadas mínimas estimadas:', execution_plan['estimated_total_calls'])
    print('Tokens de entrada estimados:', execution_plan['estimated_total_input_tokens'])

## 12. Validación cualitativa

In [ ]:
if not filtered.empty:
    audit = build_audit_sample(filtered, seed=26)
    display(audit)
    print(quality_metrics(sample, filtered, structured))
else:
    print('La auditoría se habilita cuando existen resultados reales de filtrado.')

## 13. Conclusiones y limitaciones
El diseño separa clasificación y extracción, valida todas las respuestas y conserva resultados parciales. La selección por longitud introduce sesgo hacia textos extensos; no existe ground truth y la auditoría es cualitativa. Las reseñas están en inglés, mientras que las categorías se normalizan en español. Una ejecución real requiere saldo y credencial de DeepSeek.